<a href="https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Looked at the raw CSV, not the pipeline's filled feature vector, so blanks stay NaN
instead of becoming a fake zero. 30,000 rows, 44 columns.

**Traffic counts are heavy-tailed.** `impressions_90d` has a mean of 5,200 against a
median of 731 — a ratio of 7.11, max 517,715. `sessions_90d` is the same shape at 5.30.
Plain correlation on these would be decided by a handful of giants. After `log1p` the
mean and median sit close together. That is precisely why the pipeline models the
logged column.

**Two fields are not heavy-tailed and shouldn't be treated as such.** `word_count`
(1.08) and `content_age_days` (1.09) are near-symmetric.

**`word_count` is missing on 7,699 rows — 25.7% — and the missingness is client-shaped,
not random.** 24 of 32 clients sit at exactly 0.0% missing. One client is 81.8% missing
across 7,008 rows, roughly 74% of every missing value in the column. Zero-or-broken with
almost nothing between reads as a per-client extraction failure. Two consequences: the
word-count test below speaks for 74% of the corpus with one large client mostly absent,
and filling this column with 0 downstream would make `word_count == 0` a near-clean
indicator of one client — a feature encoding client identity on a client-aware split.

**`days_since_last_update` is quantized.** Five values cover 87.8% of all rows. These
are batch timestamps, not per-page freshness. Detail in Signal 3.

**A zero in `avg_position` means "no position data," not rank zero.** 1,205 rows, 4.0%.
Described the column with those excluded.

**`ctr` is stored already multiplied by 100** — 0.76 means 0.76%. Max is exactly 100.00,
which is one click on one impression, not a miracle. Median 0.07 against a p99 of 8.33,
so per-row CTR is unusable as an average. Signal 2 weights by impressions instead.

In [10]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not IN_COLAB and os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

pd.set_option("display.width", 120)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"
print("Loaded:", df.shape)

# Working from the RAW csv (not the pipeline's fillna'd feature vector) so blanks show up as
# real NaN here, not an artificial zero -- a blind fillna(0) would inject a fake mass at 0 for
# columns like word_count where "missing" and "zero" mean different things.

fields = ["impressions_90d", "sessions_90d", "word_count", "days_since_last_update", "content_age_days"]
for col in fields:
    valid = df[col].dropna()
    n_missing = int(df[col].isna().sum())
    print(f"\n--- {col} (n={len(valid)}, missing={n_missing}) ---")
    print(valid.describe(percentiles=[.5, .75, .9, .95, .99]).to_string())
    if valid.median() > 0:
        print(f"mean/median ratio: {valid.mean() / valid.median():.2f}  (>1.5 signals a heavy right tail)")

# Heavy-tail check on the two traffic counts the pipeline log1p's -- compare raw vs. logged
for col in ["impressions_90d", "sessions_90d"]:
    raw = df[col].dropna()
    logged = np.log1p(raw)
    print(f"\n--- {col}: raw vs. log1p ---")
    print(f"raw:   mean={raw.mean():.1f}  median={raw.median():.1f}  p99={raw.quantile(.99):.1f}  max={raw.max():.1f}")
    print(f"log1p: mean={logged.mean():.2f}  median={logged.median():.2f}  p99={logged.quantile(.99):.2f}")
    print("mean/median much closer after log1p -- confirms why the pipeline models the logged column, not the raw count")

# avg_position: 0 means "no position data", not rank zero -- split before describing
no_position = df["avg_position"] == 0
print(f"\n--- avg_position ({no_position.sum()} rows / {no_position.mean() * 100:.1f}% show 0 = 'no data', excluded below) ---")
print(df.loc[~no_position, "avg_position"].describe(percentiles=[.5, .75, .9, .95, .99]).to_string())

# ctr is stored as a x100 percentage already (0.76 means 0.76%, not 76%)
print("\n--- ctr (already a x100 percentage: 0.76 = 0.76%, not 76%) ---")
print(df["ctr"].describe(percentiles=[.5, .75, .9, .95, .99]).to_string())

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Loaded: (30000, 44)

--- impressions_90d (n=30000, missing=0) ---
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
50%         731.000000
75%        3615.250000
90%       12136.400000
95%       22996.500000
99%       73505.830000
max      517715.000000
mean/median ratio: 7.11  (>1.5 signals a heavy right tail)

--- sessions_90d (n=30000, missing=0) ---
count    30000.000000
mean        37.066633
std        107.069131
min          1.000000
50%          7.000000
75%         27.000000
90%         88.000000
95%        166.000000
99%        451.010000
max       4345.000000
mean/median ratio: 5.30  (>1.5 signals a heavy right tail)

--- word_count (n=22301, missing=7699) ---
count    22301.000000
mean      3107.760325
std       1452.382598
min          8.000000
50%       2877.000000
75%       3666.000000
90%       5327.000000
95%       6173.000000
99%       7292.000000

In [11]:
print("\n=== columns ===")
print(sorted(df.columns.tolist()))

# (a) days_since_last_update: p75/p90/p95 all exactly 104 -> point mass, not a tail
print("\n--- days_since_last_update: 5 most common values ---")
vc = df["days_since_last_update"].value_counts().head(5)
print(vc.to_frame("n").assign(pct=lambda d: (d["n"] / len(df) * 100).round(1)).to_string())

spike = df["days_since_last_update"].value_counts().idxmax()
print(f"\nmost common value = {spike} days; freshness_tier it lands in:")
print(df.loc[df["days_since_last_update"] == spike, "freshness_tier"].value_counts().to_string())
print("\nshare of each freshness_tier made up by that single value:")
print(
    df.assign(is_spike=df["days_since_last_update"] == spike)
        .groupby("freshness_tier")
            .agg(n=("is_spike", "size"), spike_pct=("is_spike", lambda s: round(s.mean() * 100, 1)))
                .to_string()
                )

                # (b) word_count 25.7% missing -- random, or does it follow a category?
print(f"\n--- word_count missing overall: {df['word_count'].isna().mean() * 100:.1f}% ---")
for cat in ["content_type", "position_tier", "freshness_tier", "client_id"]:
        if cat not in df.columns or df[cat].nunique() > 15:
          continue
m = (
      df.assign(wc_missing=df["word_count"].isna())
          .groupby(cat)
              .agg(n=("wc_missing", "size"), missing_pct=("wc_missing", lambda s: round(s.mean() * 100, 1)))
                  .sort_values("missing_pct", ascending=False)
                )
print(f"\nby {cat}:")
print(m.to_string())


=== columns ===
['age_tier', 'age_tier_order', 'ai_sessions_90d', 'ai_traffic_pct', 'avg_position', 'char_count', 'char_count_tier', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'client_id', 'competition', 'competition_level', 'content_age_days', 'content_id', 'content_type', 'cpc', 'ctr', 'days_since_last_update', 'days_with_impressions', 'days_with_sessions', 'engaged_sessions_90d', 'engagement_rate', 'freshness_tier', 'impression_tier', 'impressions_90d', 'impressions_last_30d', 'impressions_prev_30d', 'main_intent', 'model_used', 'pageviews_90d', 'position_tier', 'provider_used', 'scroll_events_90d', 'scroll_rate', 'search_volume', 'sessions_90d', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct', 'users_90d', 'word_count', 'word_count_tier']

--- days_since_last_update: 5 most common values ---
                            n   pct
days_since_last_update             
20                      11573  38.6
104                      8773  29.2
22            

## 2. Signal test #1 / #2 / #3 (verdict each)

Three signals, one mini-test each, on the raw CSV. Sample floor: 50 rows per bucket.

### Signal 1 — "Longer articles earn more organic search impressions" → CONFIRMED, with a cliff

| word_count_tier | n | median impressions_90d |
|---|---|---|
| <1000 | 973 | 4 |
| 1000-2000 | 3,780 | 172 |
| 2000-3500 | 11,263 | 997 |
| 3500+ | 6,285 | 1,340 |

Monotonic, every bucket over the floor. But the shape is not a gradient. Sub-1000-word
pages sit at a median of 4 impressions — effectively invisible — and the jump to the next
tier is 43x. From 2000-3500 to 3500+ it is 1.3x. The usable claim is "get above 1,000
words," not "keep writing longer."

Caveat: `word_count_tier` is NaN on the same 7,699 rows where `word_count` is, so this
table speaks for 74% of the corpus, and the missing quarter is concentrated in one client
(see section 1). Directional, not a corpus-wide rate.

### Signal 2 — "Better average position drives higher CTR" → CONFIRMED at the ends, FALSE in the middle

Weighted CTR (total clicks / total impressions per bucket), not the mean of per-row rates.
Per-row `ctr` has a median of 0.07 and a max of 100.00 from a single impression, so
averaging it would be meaningless.

| position_tier | n | weighted CTR % |
|---|---|---|
| top_3 | 1,116 | 0.489 |
| page_1 | 11,814 | 0.350 |
| striking | 7,304 | 0.347 |
| page_3_5 | 7,242 | 0.155 |
| deep | 1,319 | 0.041 |

Monotonic overall, but page_1 and striking differ by 0.003 points across 19,000 rows.
That is not a five-step ladder. CTR separates top_3, the page_1/striking band, page_3_5,
and deep — four levels, not five.

**Data bug found here.** `position_tier == "top_3"` holds 2,321 rows, and 1,205 of them
are `avg_position == 0`, meaning no position data. The tier build did not carry over the
zero-means-no-data rule the raw column follows. 52% of the "best ranked" bucket is
unranked pages. Filtering to `avg_position > 0` removes them; the table above has that
filter applied. Fix: special-case zero in the tier build, or treat `position_tier` as
unusable without the guard.

### Signal 3 — "Content untouched for longer gets less traffic" → INSUFFICIENT DATA

The column cannot support the test. Five values cover 87.8% of 30,000 rows: 20 (38.6%),
104 (29.2%), 22 (11.9%), 8 (6.4%), 13 (1.7%). `days_since_last_update` holds roughly 20
distinct values corpus-wide. These are batch timestamps, not per-page freshness.

| freshness_tier | n | distinct values | top value | top value share |
|---|---|---|---|---|
| 0-30 | 20,480 | 16 | 20 | 56.5% |
| 31-90 | 175 | 15 | 41 | 61.1% |
| 91-180 | 9,171 | 11 | 104 | 95.7% |
| 181+ | 174 | 15 | 211 | 41.4% |

Two buckets hold 99% of rows; the other two are under 200 rows each. The `91-180` bucket
is 95.7% a single value, and its median of 1,692 impressions against 470 for `0-30` reads
as stale content outperforming fresh content by 3.6x. That is one crawl batch, not a
staleness effect.

Verdict is INSUFFICIENT DATA rather than MIXED. MIXED would claim staleness was measured
and came back ambiguous. It was not measured. Do not triage on
`days_since_last_update` until the column is rebuilt per page.

In [12]:
SAMPLE_FLOOR = 50


def trend_direction(values):
    diffs = [b - a for a, b in zip(values, values[1:])]
    if all(d > 0 for d in diffs):
        return "increasing"
    if all(d < 0 for d in diffs):
        return "decreasing"
    return "mixed"


def verdict(values, expect, n_values):
    if any(n < SAMPLE_FLOOR for n in n_values):
        return "INSUFFICIENT DATA (a bucket is under the 50-row floor)"
    direction = trend_direction(values)
    if direction == "mixed":
        return "MIXED"
    spread = abs(values[-1] - values[0]) / (abs(values[0]) if values[0] else 1)
    if direction == expect and spread > 0.05:
        return "CONFIRMED"
    if direction == expect:
        return "FALSE (direction matches but the gap is too small to call a real effect)"
    return "OPPOSITE"


# --- Signal 1: "Longer articles earn more organic search impressions" (word_count vs impressions_90d)
wc_order = ["<1000", "1000-2000", "2000-3500", "3500+"]
g1 = (
    df.dropna(subset=["word_count_tier"])
    .groupby("word_count_tier")["impressions_90d"]
    .agg(n="count", median_impressions="median")
    .reindex(wc_order)
)
print("--- Signal 1: word_count_tier vs. median impressions_90d ---")
print(g1.to_string())
v1 = verdict(g1["median_impressions"].tolist(), "increasing", g1["n"].tolist())
print(f"VERDICT: {v1}")

# --- Signal 2: "Better average position drives higher CTR" (position_tier vs. weighted ctr)
# weighted, not mean-of-per-row-rates: total_clicks / total_impressions per bucket
#
# Data-quality catch: ALL 1,205 avg_position==0 ("no position data") rows are mislabeled into
# position_tier=="top_3" upstream -- the tier build did not special-case the zero-means-no-data
# rule the way the raw avg_position column does. Filtering to avg_position > 0 below removes
# them; skipping that filter would falsely count 1,205 unranked pages as best-ranked.
mislabeled = ((df["position_tier"] == "top_3") & (df["avg_position"] == 0)).sum()
top3_total = (df["position_tier"] == "top_3").sum()
print(f"position_tier=='top_3': {top3_total} rows, {mislabeled} of them are actually avg_position==0 (no data)")

pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
g2 = (
    df[df["avg_position"] > 0]
    .groupby("position_tier")
    .agg(n=("ctr", "count"), total_impressions=("impressions_90d", "sum"), total_clicks=("clicks_90d", "sum"))
    .reindex(pos_order)
)
g2["weighted_ctr"] = g2["total_clicks"] / g2["total_impressions"] * 100
print("\n--- Signal 2: position_tier (best -> worst) vs. weighted CTR ---")
print(g2[["n", "weighted_ctr"]].to_string())
v2 = verdict(g2["weighted_ctr"].tolist(), "decreasing", g2["n"].tolist())
print(f"VERDICT: {v2}")

# --- Signal 3: "Content untouched for longer gets less traffic" (freshness_tier vs impressions_90d)
# PRE-TEST: is days_since_last_update actually measuring per-page freshness?
top5 = df["days_since_last_update"].value_counts().head(5)
print("\n--- Signal 3 pre-test: days_since_last_update concentration ---")
print(top5.to_frame("n").assign(pct=(top5 / len(df) * 100).round(1)).to_string())
print(f"top 5 values cover {top5.sum() / len(df) * 100:.1f}% of all rows")

fresh_order = ["0-30", "31-90", "91-180", "181+"]
dom = (
    df.groupby("freshness_tier")["days_since_last_update"]
    .agg(n="count", n_unique="nunique",
         top_value=lambda s: s.value_counts().idxmax(),
         top_value_pct=lambda s: round(s.value_counts().iloc[0] / len(s) * 100, 1))
    .reindex(fresh_order)
)
print("\nper-bucket: is each bucket one repeated value?")
print(dom.to_string())

g3 = (
    df.groupby("freshness_tier")["impressions_90d"]
    .agg(n="count", median_impressions="median")
    .reindex(fresh_order)
)
print("\n--- Signal 3: freshness_tier vs. median impressions_90d ---")
print(g3.to_string())
v3 = "INSUFFICIENT DATA -- days_since_last_update is quantized to a few batch values; buckets measure crawl batch, not page freshness"
print(f"VERDICT: {v3}")

--- Signal 1: word_count_tier vs. median impressions_90d ---
                     n  median_impressions
word_count_tier                           
<1000              973                 4.0
1000-2000         3780               172.0
2000-3500        11263               997.0
3500+             6285              1340.0
VERDICT: CONFIRMED
position_tier=='top_3': 2321 rows, 1205 of them are actually avg_position==0 (no data)

--- Signal 2: position_tier (best -> worst) vs. weighted CTR ---
                   n  weighted_ctr
position_tier                     
top_3           1116      0.488544
page_1         11814      0.350324
striking        7304      0.346876
page_3_5        7242      0.154905
deep            1319      0.041359
VERDICT: CONFIRMED

--- Signal 3 pre-test: days_since_last_update concentration ---
                            n   pct
days_since_last_update             
20                      11573  38.6
104                      8773  29.2
22                       3564  11.9


## 3. The flag-linked test

Testing `page_one_decay_risk` from `scripts/02_baseline_score.py`:

row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180

The rule bets that among page-1 content, older pages decline more often than younger
page-1 content. That is the "decay risk" the flag is named for. Note the rule already
guards `avg_position > 0`, so it avoids the tier bug from Signal 2. Mirrored that guard
in the test.

**Result: OPPOSITE.**

| age group | n | declining rate |
|---|---|---|
| old (≥180d, flagged) | 7,076 | 51.8% |
| young (<180d, not flagged) | 5,907 | 61.7% |

Gap: −9.9 points, the wrong way. Page-1 base rate is 56.3% against 54.2% corpus-wide.

**The flag picks the right population and then splits it backwards.** Page-1 content does
decline slightly more than average, so filtering to page 1 is sound. The age condition
then selects the half that declines *less*. Reviewers working this queue top-down are
being pointed at the more stable half.

**Checked whether the gap is real or a proxy for something else.** Four cuts, all
negative:

| slice | old | young | gap |
|---|---|---|---|
| impressions Q1 (low) | 36.2% | 47.1% | −10.9 |
| impressions Q2 | 68.7% | 73.8% | −5.1 |
| impressions Q3 | 60.8% | 69.7% | −8.9 |
| impressions Q4 (high) | 46.0% | 51.8% | −5.8 |

Not a traffic-volume artifact. Then checked the freshness confound: old pages are 37.8%
in the `91-180` bucket versus 16.4% for young, and Signal 3 showed that bucket is 95.7%
one crawl batch. Inside a single batch the gap holds and widens:

| freshness_tier | old | young | gap |
|---|---|---|---|
| 0-30 (n=9,189) | 46.4% | 60.3% | −13.9 |
| 91-180 (n=3,644) | 60.7% | 68.9% | −8.2 |

Also split on `clicks_90d == 0` (35.7% of page-1 rows): gap −9.1 and −11.5. Narrow claim
only — this shows the gap is not produced by the zero-click subgroup in this CSV. It is
not a check on the warehouse label leak documented in `w03_feature_leakage_check.ipynb`,
which concerns a different column on a different dataset.

Eight slices, every one negative, none under the 50-row floor. The age term is
anti-predictive here, not confounded.

**Standing caveat.** This grades the rule against `trend_direction`, the CSV label. It
measures whether the flag agrees with the label the pipeline scores against. It is not a
measurement of true traffic decline.

**Fix:** invert the age condition to `content_age_days < 180`, or drop the age term and
flag on the page-1 filter alone. Both are one-line changes to
`scripts/02_baseline_score.py`. Verify against a held-out client before shipping — this
is one dataset.

In [15]:
# FlyRank's real flag being tested: `page_one_decay_risk` from scripts/02_baseline_score.py --
#   if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180
# Assumption behind the rule: among page-1 content, OLDER pages are more likely to be declining
# than younger page-1 content -- that's the "decay risk" the flag is betting on.
#
# Note this rule already guards avg_position > 0 -- it dodges the position_tier "no data
# mislabeled as top_3" trap found in section 2. Mirroring that guard here.

page_one = df[(df["avg_position"] > 0) & (df["avg_position"] <= 10)].copy()
page_one["age_group"] = np.where(page_one["content_age_days"] >= 180, "old (>=180d, flagged)", "young (<180d, not flagged)")

flag_test = page_one.groupby("age_group")["trend_direction"].apply(
    lambda s: (s.str.lower() == "down").mean()
).rename("declining_rate")
flag_n = page_one.groupby("age_group").size().rename("n")
flag_table = pd.concat([flag_n, flag_test], axis=1)
print("--- page_one_decay_risk: page-1 content (avg_position 0-10), old vs. young ---")
print(flag_table.to_string())

overall_base_rate = (df["trend_direction"].str.lower() == "down").mean()
page_one_base = (page_one["trend_direction"].str.lower() == "down").mean()
print(f"\nbase rate -- page-1 slice: {page_one_base * 100:.1f}%  |  all rows: {overall_base_rate * 100:.1f}%")
print("the page-1 slice is the right comparison for a page-1 flag")

old_rate = flag_table.loc["old (>=180d, flagged)", "declining_rate"]
young_rate = flag_table.loc["young (<180d, not flagged)", "declining_rate"]
old_n = flag_table.loc["old (>=180d, flagged)", "n"]
young_n = flag_table.loc["young (<180d, not flagged)", "n"]

if old_n < SAMPLE_FLOOR or young_n < SAMPLE_FLOOR:
    flag_verdict = "INSUFFICIENT DATA (a group is under the 50-row floor)"
elif old_rate - young_rate > 0.05:
    flag_verdict = "CONFIRMED"
elif young_rate - old_rate > 0.05:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "FALSE (gap too small to call a real effect)"

print(f"\nold declining rate {old_rate * 100:.1f}% (n={old_n}) vs. young declining rate {young_rate * 100:.1f}% (n={young_n})")
print(f"gap (old - young): {(old_rate - young_rate) * 100:+.1f} points")
print(f"VERDICT: {flag_verdict}")

--- page_one_decay_risk: page-1 content (avg_position 0-10), old vs. young ---
                               n  declining_rate
age_group                                       
old (>=180d, flagged)       7076        0.518089
young (<180d, not flagged)  5907        0.617064

base rate -- page-1 slice: 56.3%  |  all rows: 54.2%
the page-1 slice is the right comparison for a page-1 flag

old declining rate 51.8% (n=7076) vs. young declining rate 61.7% (n=5907)
gap (old - young): -9.9 points
VERDICT: OPPOSITE


In [13]:
pg = page_one.copy()
pg["declining"] = pg["trend_direction"].str.lower().eq("down")

# (a) does the age gap survive INSIDE impression quartiles?
pg["imp_q"] = pd.qcut(pg["impressions_90d"], 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
t = (pg.groupby(["imp_q", "age_group"], observed=True)
       .agg(n=("declining", "size"), declining_rate=("declining", "mean")))
t["declining_rate"] = (t["declining_rate"] * 100).round(1)
print("--- (a) old vs young WITHIN impression quartile ---")
print(t.to_string())

# (b) is 'old' just a proxy for the days_since_last_update point mass?
print("\n--- (b) freshness_tier mix by age group (row-normalised) ---")
print(pd.crosstab(pg["age_group"], pg["freshness_tier"], normalize="index").round(3).to_string())

# (c) correct baseline
print(f"\n--- (c) page-1 base rate: {pg['declining'].mean() * 100:.1f}%"
      f"  |  all rows: {overall_base_rate * 100:.1f}% ---")

# (d) zero-click rows -- label may be non-declining by construction (w03 finding)
if "clicks_90d" in pg.columns:
    pg["zero_clicks"] = pg["clicks_90d"] == 0
    print(f"\n--- (d) page-1 rows with clicks_90d == 0: {pg['zero_clicks'].sum()}"
          f" ({pg['zero_clicks'].mean() * 100:.1f}%) ---")
    print(pg.groupby(["zero_clicks", "age_group"], observed=True)
            .agg(n=("declining", "size"), declining_rate=("declining", "mean")).round(3).to_string())

--- (a) old vs young WITHIN impression quartile ---
                                       n  declining_rate
imp_q   age_group                                       
Q1 low  old (>=180d, flagged)       1929            36.2
        young (<180d, not flagged)  1330            47.1
Q2      old (>=180d, flagged)       1487            68.7
        young (<180d, not flagged)  1747            73.8
Q3      old (>=180d, flagged)       1778            60.8
        young (<180d, not flagged)  1466            69.7
Q4 high old (>=180d, flagged)       1882            46.0
        young (<180d, not flagged)  1364            51.8

--- (b) freshness_tier mix by age group (row-normalised) ---
freshness_tier               0-30   181+  31-90  91-180
age_group                                              
old (>=180d, flagged)       0.606  0.013  0.003   0.378
young (<180d, not flagged)  0.830  0.000  0.005   0.164

--- (c) page-1 base rate: 56.3%  |  all rows: 54.2% ---

--- (d) page-1 rows with clicks_90

In [14]:
# Does the age gap survive INSIDE a single freshness batch?
# (b) showed old/young differ 37.8% vs 16.4% in the 91-180 bucket, which is 95.7% one value.
for tier in ["0-30", "91-180"]:
    sub = pg[pg["freshness_tier"] == tier]
    t = (sub.groupby("age_group", observed=True)
           .agg(n=("declining", "size"), declining_rate=("declining", "mean")))
    t["declining_rate"] = (t["declining_rate"] * 100).round(1)
    print(f"\n--- freshness_tier == {tier} (n={len(sub)}) ---")
    print(t.to_string())
    if len(t) == 2 and t["n"].min() >= 50:
        gap = t.loc["old (>=180d, flagged)", "declining_rate"] - t.loc["young (<180d, not flagged)", "declining_rate"]
        print(f"gap (old - young): {gap:+.1f} points")
    else:
        print("below floor or single group -- no verdict")


--- freshness_tier == 0-30 (n=9189) ---
                               n  declining_rate
age_group                                       
old (>=180d, flagged)       4285            46.4
young (<180d, not flagged)  4904            60.3
gap (old - young): -13.9 points

--- freshness_tier == 91-180 (n=3644) ---
                               n  declining_rate
age_group                                       
old (>=180d, flagged)       2673            60.7
young (<180d, not flagged)   971            68.9
gap (old - young): -8.2 points


## 4. What this means in practice


Three things a content team should take from this, in order of what it costs to ignore.

**Stop using `page_one_decay_risk` as a review queue.** It has the direction backwards.
Page-1 pages older than 180 days decline 51.8% of the time; younger page-1 pages decline
61.7%. Reviewers working that queue top-down are spending their time on the more stable
half. The page-1 filter itself is sound — invert or drop the age condition.

**Do not triage on `days_since_last_update`.** The column holds about 20 distinct values
across 30,000 rows, so it records crawl batches rather than page freshness. Any ranking
built on it sorts by batch. Rebuild it per page before it is used for anything.

**Two fields need a guard before you read them.** `position_tier == "top_3"` is 52%
unranked pages — check `avg_position > 0` first. And `word_count` is missing on a quarter
of the corpus, concentrated in one client, so filling it with 0 downstream turns it into a
client identifier rather than a content measure.

What does hold: word count and position. Getting above 1,000 words matters a great deal
and past that the returns flatten; position improvements compound into CTR at the top and
bottom of the range, though page_1 and striking are indistinguishable from each other.

All of this is measured against `trend_direction`, the CSV's own label, on one dataset.
Directional and decision-support, not a forecast.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.